# Codeforces Round 932 (Div. 2)

- https://codeforces.com/contest/1935

`-` 구현 속도가 느려 D번을 풀지 못했다. 20분만 더 있었더라면....

`-` 그래도 긍정적으로 생각할 수 있는 점은 문제를 접근하는 데 무리가 없었다는 점이다. 사실 이렇게 오래 걸린 거면 문제가 있는 것 같긴 한데 아무튼 아이디어를 떠올리지 못해 못푼 건 아니다

`-` 단지, A, B, C번을 구현하는 데 고려할 게 많았을 뿐이다 (A번은 내 실수긴 하지만)

`-` 구현량이 많아지니까 뇌가 멈춰버린다. 아이디어를 정리하면서 어떻게 구현할지도 같이 정리해서 우왕좌왕하는 빈도를 줄이자 (이는 특히 구현 방법이 여러 개일 때 큰 도움이 된다)

## A. Entertainment in MAC (00:06)

`-` $n$은 항상 짝수인데 문제 읽다가 망각해서 콘테스트 땐 자연수로 일반화해서 풀었다. (스스로 많은 조건 분기를 만들어 버림;)

`-` $s$를 뒤집은 문자열을 $s'$이라 하자. $s \le s'$이라면 $2$번 연산을 $n$번 수행하자. $n$이 짝수이므로 이는 $s$와 같다

`-` 그렇지 않다면 $1$번 연산을 $n-1$번 수행한 뒤 $2$번 연산을 수행하면 된다. 이는 $s' + s$와 같다

In [1]:
def solve_testcase(s):
    if s <= s[::-1]:
        return s
    return s[::-1] + s


def solution():
    t = int(input())
    for _ in range(t):
        n = int(input())
        s = input().rstrip()
        answer = solve_testcase(s)
        print(answer)


solution()

# input
# 1
# 4
# cpm

 1
 4
 cpm


cpm


`-` 에디토리얼 보고 $n$이 항상 짝수인 걸 깨달았다...

## B. Informatics in MAC (00:36)

`-` 배열에 $0$이 없으면 아무렇게나 둘로 나누면 정답이다. 그렇지 않고 배열에 $0$이 존재하면 $0$을 포함하는 세그먼트가 존재할 것이고 그럼 MEX가 $1$이 된다. 따라서 모든 세그먼트의 MEX는 $1$이어야 한다

`-` 또, 배열에 $1$이 존재한다면 각 세그먼트엔 $0$이 존재하므로 $1$을 포함하는 세그먼트의 MEX는 $2$가 된다. 따라서 모든 세그먼트의 MEX는 $2$여야 한다. 여기서 알 수 있는 점은 전체 배열의 MEX가 모든 세그먼트의 MEX가 된다는 것이다

`-` 동일한 MEX를 가지는 세그먼트를 합쳐서 하나로 만들어도 MEX는 변하지 않는다. 따라서 $k>2$라면 항상 $k = 2$로 만들 수 있다

`-` 배열의 MEX를 계산한 뒤 해당 MEX를 가지도록 앞에서부터 그리디하게 세그먼트를 구성하자. 그 후 남은 세그먼트의 MEX가 동일한지 판단하면 된다

In [9]:
def compute_mex(array, max_value):
    seen = [False] * (max_value + 1)
    for a in array:
        seen[a] = True
    mex = 0
    while mex <= max_value and seen[mex]:
        mex += 1
    return mex


def solve_testcase(array):
    n = len(array)
    mex = compute_mex(array, n - 1)
    subsegments = []
    seen = [False] * mex
    count = 0
    for i, a in enumerate(array):
        if a < mex and not seen[a]:
            seen[a] = True
            count += 1
        if count == mex:
            subsegments.append((1, i + 1))
            break
    subsegment_mex = compute_mex(array[i + 1:], n - 1)
    if subsegment_mex == mex:
        subsegments.append((i + 2, n))
        return subsegments
    return -1


def solution():
    t = int(input())
    for _ in range(t):
        n = int(input())
        a = list(map(int, input().split()))
        subsegments = solve_testcase(a)
        if subsegments == -1:
            print(-1)
            continue
        print(len(subsegments))
        for l, r in subsegments:
            print(l, r)


solution()

# input
# 1
# 8
# 0 1 7 1 0 1 0 3

 1
 8
 0 1 7 1 0 1 0 3


2
1 2
3 8


`-` 인사이트가 중요한 문제였다. 구현하는 게 은근 오래 걸렸다

## C. Messenger in MAC (01:43)

`-` 길이가 $k$인 임의의 메시지 집합에 대해 $b$의 최솟값 $b_{\min}$과 최댓값 $b_{\max}$를 고려하자. 공식에 의해 $\sum\limits_{i=1}^{k-1}|b_{p_i} - b_{p_{i+1}}| = b_{\max} - b_{\min}$이다

`-` 따라서 $b_{\min}$과 $b_{\max}$가 고정됐다면 메시지를 읽는 데 걸리는 시간을 그리디하게 계산할 수 있다. 단순히 $a$를 기준으로 오름차순 정렬한 뒤 소요 시간이 $l$을 넘지 않을 때까지 메시지를 읽으면 된다. 하지만 이는 $O\left(n^3 \log n\right)$이라 시간 초과이다. 이전 정보를 재사용함으로써 시간 복잡도를 줄여보자

`-` 이를 위해 우선 $p$를 $b$를 기준으로 오름차순 정렬하자. 그리고 시작 메시지를 고정시킨 뒤 최대로 읽을 수 있는 메시지 개수를 구할 것이다

`-` 현재 $i$번째 메시지를 읽을지 말지 고려하고 있다고 해보자. 읽어도 소요 시간을 초과하지 않는다면 읽는 게 이득이다. 만약 소요 시간을 초과한다고 해도 읽자. 그리고 여태까지 읽은 메시지 중 $a$가 가장 큰 것을 읽지 말자 (우선순위 큐를 사용하자). 이를 소요 시간을 초과하지 않을 때까지 반복하면 된다. 만약 $i$번째 메시지를 읽는 게 손해였다면 어차피 읽지 않도록 설계되니 문제 없다

`-` $b$에 의한 소요 시간은 $b_i - b_{\min}$이며 $a$에 의한 소요 시간은 우선순위 큐에 존재하는 $a$의 합이다 (매번 합을 새로 구할 필요 없이 따로 변수를 선언해 관리하자). 둘의 합이 총 소요 시간이 된다

`-` 이때 $i$번째 메시지를 읽었는데 $a$가 너무 커서 읽지 않기로 한 경우 $b$에 의한 소요 시간이 $b_i - b_{\min}$로 실제로 읽은 메시지에 의한 것보다 커지게 된다. 근데 이는 상관없다. 왜냐하면 새로운 메시지를 읽는 경우 어차피 $b$에 의한 소요 시간에 $b_i$가 영향을 끼치지 않기 때문이다. 만약 새로운 메시지를 읽지 않는다면 당연하게도 여태까지 최대로 읽은 메시지 개수가 변하지 않으므로 상관없는 건 매한가지다

`-` 스위핑으로 처리하면 고정된 시작 메시지에 대해 최대로 읽을 수 있는 메시지 개수를 $O(n\log n)$에 구할 수 있다. 따라서 전체 알고리즘의 시간 복잡도는 $O\left(n^2\log n\right)$이다

In [4]:
import heapq


def maximize_size(array, start_index, limit):
    n = len(array)
    b_min = array[start_index][1]
    pq = []
    cost = 0
    max_size = 0
    for i in range(start_index, n):
        a, b = array[i]
        heapq.heappush(pq, -a)
        cost += a
        b_cost = b - b_min
        while pq and cost + b_cost > limit:
            cost -= -heapq.heappop(pq)
        max_size = max(len(pq), max_size)
    return max_size


def solve_testcase(array, limit):
    n = len(array)
    array.sort(key=lambda x: x[1])
    max_size = 0
    for start_index in range(n):
        size = maximize_size(array, start_index, limit)
        if max_size < size:
            max_size = size
    return max_size    


def solution():
    t = int(input())
    for _ in range(t):
        n, l = map(int, input().split())
        p = [list(map(int, input().split())) for _ in range(n)]
        answer = solve_testcase(p, l)
        print(answer)


solution()

# input
# 1
# 3 12
# 4 8
# 2 1
# 2 12

 1
 3 12
 4 8
 2 1
 2 12


2


`-` 변수명을 잘못 적는 뻘짓을 해서 20분을 내다 버린 슬픈 상황이 있었다

`-` 에디토리얼 보고 리팩토링 했다. 기존 코드는 쓸데없이 너무 복잡했다 (나는 새로운 $a$를 넣기 전에 $l$을 초과하지 않도록 조작을 했다. 만약 $a$가 힙의 최댓값보다 크다면 패스)

## D. Exam in MAC (Upsolving)

`-` 문제의 조건을 만족하는 $(x,y)$ 쌍을 바로 구하려고 하니까 쉽지 않다. 대신 포함-배제의 원리를 이용하자

`-` 표본 공간 $\Omega = \{(x,y) \mid x \in \mathbb{N}_0,\, y \in \mathbb{N}_0,\, 0 \le x \le y \le c \}$에 대해 $\Omega$의 부분집합을 다음과 같이 정의하자

`-` $A = \{(x,y) \in \Omega \mid x+y \notin s,\, y-x \notin s\}$

`-` $B = \{(x,y) \in \Omega \mid x+y \in s\}$

`-` $C = \{(x,y) \in \Omega \mid y-x \in s\}$

`-` $D = \{(x,y) \in \Omega \mid x+y \in s,\, y-x \in s\}$

`-` $A^c = \{(x,y) \in \Omega \mid (x+y \in s \text{ 또는 } y-x \in s)\}$

`-` 위와 같이 집합을 설정하면 포함-배제의 원리에 따라 $|A^c| = |B| + |C| - |D|$가 성립한다. 따라서 $|A| = |\Omega| - |A^c| = |\Omega| - (|B| + |C| - |D|)$이다

`-` $|A|$를 계산하기 위해 $|\Omega|, |B|, |C|, |D|$를 알아야 한다

`-` 먼저 $|\Omega|$를 계산하자. 고정된 $y$에 대해 가능한 $x$의 수는 $y+1$이다. $0\le y \le c$이므로 $|\Omega| = 1 + 2 + \cdots + c + 1 = \frac{(c+1)(c+2)}{2}$이다

`-` 이제 $|B|$를 계산하자. 집합 $s$의 원소 $s_i$에 대해 $x+y = s_i$를 만족하는 $(x,y)$ 쌍은 $(0, s_i),\, (1, s_i - 1),\, \dots,\, (\left\lfloor\frac{s_i}{2}\right\rfloor, s_i - \left\lfloor\frac{s_i}{2}\right\rfloor)$로 총 $\left\lfloor\frac{s_i}{2}\right\rfloor + 1$개이다. 따라서 $|B| = \sum\limits_{i=1}^{n} \left(\left\lfloor\frac{s_i}{2}\right\rfloor + 1\right)$이다

`-` $|C|$도 비슷하게 구할 수 있다. $y - x = s_i$를 만족하는 $(x,y)$ 쌍은 $(0, s_i),\, (1, s_i + 1),\, \dots,\, (c - s_i, c)$로 총 $c - s_i + 1$개이다. 따라서 $|C| = \sum\limits_{i=1}^{n} (c - s_i + 1)$이다

`-` 마지막으로 $|D|$만 구하면 끝이다. 먼저 $x+y$와 $y-x$의 홀짝성은 같아야 한다 (다르면 모순). $x+y = s_i$일 때 가능한 $y-x$는 $s_i$가 짝수인 경우 $0$과 $s_i$ 사이의 모든 짝수이고 $s_i$가 홀수라면 $0$과 $s_i$ 사이의 모든 홀수이다. 이때 $y-x$가 $s$의 원소여야 한다

`-` $s$의 원소를 짝수, 홀수에 따라 분리하자. 그리고 오름차순으로 정렬하면 $s$의 원소 중 $s_i$를 넘지 않는 짝수 또는 홀수의 수를 이분 탐색으로 $O(\log n)$에 구할 수 있다

`-` 결과적으로 위의 과정을 통해 $|A|$를 $O(n \log n)$에 계산할 수 있다

In [10]:
from bisect import bisect_right


def solve_testcase(s, c):
    evens = sorted([s_i for s_i in s if s_i % 2 == 0])
    odds = sorted([s_i for s_i in s if s_i % 2 == 1])
    total = ((c + 1) * (c + 2)) // 2
    plus = minus = both = 0
    for s_i in s:
        plus += s_i // 2 + 1
        minus += c - s_i + 1
        if s_i % 2 == 0:
            both += bisect_right(evens, s_i)
        else:
            both += bisect_right(odds, s_i)
    return total - (plus + minus - both)


def solution():
    t = int(input())
    for _ in range(t):
        n, c = map(int, input().split())
        s = list(map(int, input().split()))
        answer = solve_testcase(s, c)
        print(answer)


solution()

# input
# 1
# 3 3
# 1 2 3

 1
 3 3
 1 2 3


3
